# 学習 v2（類似度特徴量あり）

v1（learning.ipynb）からの変更点：
- CVの評価基準をベイスギ（樹種15）除外に変更
- trainの各樹種平均スペクトルとの距離を特徴量に追加
- 上記2点を踏まえてalphaを再チューニング

**入力ファイル（`models/` フォルダ）**

| ファイル | 内容 |
|---|---|
| `models/X_train.npy` | 前処理済みtrain特徴量（PCA済み） |
| `models/y_train.npy` | log変換済みtrain目的変数 |

**出力ファイル（`models/` フォルダ）**

| ファイル | 内容 |
|---|---|
| `models/model_v2.pkl` | 学習済みRidgeモデル（類似度特徴量あり） |
| `models/species_centroids.npy` | train各樹種の平均スペクトル（predicting_v2で使用） |
| `models/species_centroid_ids.npy` | 樹種番号の対応表 |

## 1. ライブラリの読み込み

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.spatial.distance import cdist

# Noto Sans CJK JP を指定（環境に既存のフォント）
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['figure.dpi'] = 120

print('ライブラリ読み込み完了')

ライブラリ読み込み完了


## 2. データの読み込み

In [3]:
# PCA済み特徴量・目的変数を読み込む
X_train = np.load('models/X_train.npy')  # shape: (1322, 20)
y_train = np.load('models/y_train.npy')  # log変換済み

# SNV済みの生スペクトルも読み込む（類似度特徴量の計算に使う）
train_raw = pd.read_csv('../data/train.csv', encoding='shift-jis')
train_raw.columns = (
    ['sample_number', 'species_number', 'species_name', 'moisture_content']
    + list(train_raw.columns[4:])
)
spec_cols = list(train_raw.columns[4:])
groups = train_raw['species_number'].values

def snv(X):
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

X_snv = snv(train_raw[spec_cols].values)

print(f'X_train（PCA済み）: {X_train.shape}')
print(f'y_train:           {y_train.shape}')
print(f'X_snv（生スペクトル）: {X_snv.shape}')
print(f'樹種の種類数: {len(np.unique(groups))}')

X_train（PCA済み）: (1322, 20)
y_train:           (1322,)
X_snv（生スペクトル）: (1322, 1555)
樹種の種類数: 13


## 3. 類似度特徴量とは

trainの各樹種の「平均スペクトル」を基準点（centroid）として、
各サンプルのスペクトルがそれぞれの基準点からどれだけ離れているか（ユークリッド距離）を計算する。

13樹種あれば13個の距離が得られ、これを新しい特徴量としてPCAの20次元に追加する（合計33次元）。

**なぜ有効か**：testは未知樹種だが「trainのどの樹種に似ているか」という文脈情報を与えることで、
モデルが似た樹種の傾向を参考にしながら予測できるようになる可能性があるため。

**ルール適合性**：基準点はtrainだけから作るので、testサンプル同士の情報は使わない。
1サンプルが与えられたときにtrainとの距離を計算するだけなので問題なし。

In [4]:
def make_similarity_features(X, X_train_ref, groups_ref):
    """
    各サンプルとtrainの各樹種平均スペクトルとのユークリッド距離を計算する

    Parameters
    ----------
    X          : 距離を計算したいサンプル群 (n_samples, n_wavelengths)
    X_train_ref: 基準となるtrainスペクトル  (n_train, n_wavelengths)
    groups_ref : X_train_refの樹種番号      (n_train,)

    Returns
    -------
    distances  : (n_samples, n_species) の距離行列
    """
    species_list = sorted(np.unique(groups_ref))
    centroids = np.array([
        X_train_ref[groups_ref == s].mean(axis=0)
        for s in species_list
    ])
    return cdist(X, centroids, metric='euclidean')

# 動作確認
sample_sim = make_similarity_features(X_snv[:3], X_snv, groups)
print(f'類似度特徴量のshape（サンプル3件）: {sample_sim.shape}')
print(f'  → 13樹種それぞれとの距離が得られる')

類似度特徴量のshape（サンプル3件）: (3, 13)
  → 13樹種それぞれとの距離が得られる


## 4. バリデーション戦略

v1からの変更点：**ベイスギ（樹種15）をCVから除外する**

理由：ベイスギはスペクトルが他樹種と大きく異なり（PCA第1主成分が+9.7と突出）、
testの6樹種には同様の樹種がいない。
ベイスギを含めるとCVスコアが悪く見えすぎて、改善の判断精度が下がるため。

※最終学習はベイスギを含む全データで行う（学習から外す理由はない）

In [5]:
gkf = GroupKFold(n_splits=5)

# ベイスギ除外マスク
mask_cv = groups != 15
X_train_cv = X_train[mask_cv]
X_snv_cv   = X_snv[mask_cv]
y_train_cv = y_train[mask_cv]
groups_cv  = groups[mask_cv]

print('GroupKFold の分割確認（ベイスギ除外、各foldのvalidation樹種）')
print('-' * 55)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_cv, y_train_cv, groups_cv)):
    val_species = sorted(np.unique(groups_cv[val_idx]))
    print(f'Fold {fold+1}: validation樹種={val_species}  ({len(val_idx)}サンプル)')

GroupKFold の分割確認（ベイスギ除外、各foldのvalidation樹種）
-------------------------------------------------------
Fold 1: validation樹種=[np.int64(3), np.int64(12)]  (270サンプル)
Fold 2: validation樹種=[np.int64(11), np.int64(14)]  (229サンプル)
Fold 3: validation樹種=[np.int64(4), np.int64(5)]  (201サンプル)
Fold 4: validation樹種=[np.int64(13), np.int64(17), np.int64(19)]  (251サンプル)
Fold 5: validation樹種=[np.int64(1), np.int64(8), np.int64(16)]  (259サンプル)


## 5. alphaのチューニング

In [6]:
alphas = [700, 800, 900, 1000, 1100, 1200, 1300, 1500]
results = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    rmse_log_folds  = []
    rmse_orig_folds = []

    for tr_idx, val_idx in gkf.split(X_train_cv, y_train_cv, groups_cv):
        # 類似度特徴量をfold内trainだけで作る（データリーク防止）
        sim_tr  = make_similarity_features(X_snv_cv[tr_idx],  X_snv_cv[tr_idx], groups_cv[tr_idx])
        sim_val = make_similarity_features(X_snv_cv[val_idx], X_snv_cv[tr_idx], groups_cv[tr_idx])

        # PCA特徴量 + 類似度特徴量を結合
        Xtr  = np.hstack([X_train_cv[tr_idx],  sim_tr])
        Xval = np.hstack([X_train_cv[val_idx], sim_val])

        model.fit(Xtr, y_train_cv[tr_idx])

        y_pred_log  = model.predict(Xval)
        y_pred_orig = np.expm1(y_pred_log)
        y_val_orig  = np.expm1(y_train_cv[val_idx])

        rmse_log_folds.append(
            np.sqrt(mean_squared_error(y_train_cv[val_idx], y_pred_log))
        )
        rmse_orig_folds.append(
            np.sqrt(mean_squared_error(y_val_orig, y_pred_orig))
        )

    results.append({
        'alpha'          : alpha,
        'rmse_log_mean'  : np.mean(rmse_log_folds),
        'rmse_log_std'   : np.std(rmse_log_folds),
        'rmse_orig_mean' : np.mean(rmse_orig_folds),
        'rmse_orig_std'  : np.std(rmse_orig_folds),
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))
print()

best_row = results_df.loc[results_df['rmse_orig_mean'].idxmin()]
BEST_ALPHA = int(best_row['alpha'])
print(f'最良のalpha: {BEST_ALPHA}  (CV RMSE={best_row["rmse_orig_mean"]:.4f})')

 alpha  rmse_log_mean  rmse_log_std  rmse_orig_mean  rmse_orig_std
   700         0.4241        0.1666         17.9934         5.8463
   800         0.4238        0.1686         17.9209         5.9830
   900         0.4239        0.1703         17.8740         6.0985
  1000         0.4243        0.1719         17.8473         6.1989
  1100         0.4249        0.1732         17.8367         6.2885
  1200         0.4258        0.1744         17.8394         6.3706
  1300         0.4268        0.1755         17.8529         6.4474
  1500         0.4294        0.1772         17.9064         6.5913

最良のalpha: 1100  (CV RMSE=17.8367)


## 6. 最終モデルの学習

ベストalphaで**全trainデータ（ベイスギ含む）**を使って再学習する。
類似度特徴量もtrain全体から計算した樹種平均を使う。

In [7]:
# train全体の類似度特徴量
sim_train_all = make_similarity_features(X_snv, X_snv, groups)
X_train_final = np.hstack([X_train, sim_train_all])

final_model = Ridge(alpha=BEST_ALPHA)
final_model.fit(X_train_final, y_train)

# train全体での参考RMSE
y_pred_train_log  = final_model.predict(X_train_final)
y_pred_train_orig = np.expm1(y_pred_train_log)
y_train_orig      = np.expm1(y_train)
train_rmse = np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))

print(f'最終モデル（alpha={BEST_ALPHA}）')
print(f'特徴量次元数: {X_train_final.shape[1]}  （PCA20次元 + 類似度13次元）')
print(f'train RMSE（参考値）: {train_rmse:.4f}')

最終モデル（alpha=1100）
特徴量次元数: 33  （PCA20次元 + 類似度13次元）
train RMSE（参考値）: 25.3501


## 7. モデルと樹種centroids の保存

predicting_v2.ipynb で類似度特徴量を再現するために、
train全体から計算した樹種平均スペクトル（centroids）も保存する。

In [8]:
# 樹種平均スペクトル（centroids）を保存
species_list = sorted(np.unique(groups))
centroids = np.array([
    X_snv[groups == s].mean(axis=0)
    for s in species_list
])

joblib.dump(final_model, 'models/model_v2.pkl')
np.save('models/species_centroids.npy',  centroids)
np.save('models/species_centroid_ids.npy', np.array(species_list))

print('保存完了')
print(f'  models/model_v2.pkl             学習済みモデル')
print(f'  models/species_centroids.npy    樹種平均スペクトル {centroids.shape}')
print(f'  models/species_centroid_ids.npy 樹種番号 {species_list}')
print()
print(f'  アルゴリズム : Ridge回帰（類似度特徴量あり）')
print(f'  alpha        : {BEST_ALPHA}')
print(f'  CV RMSE      : {best_row["rmse_orig_mean"]:.4f} ± {best_row["rmse_orig_std"]:.4f}')
print()
print('次は predicting_v2.ipynb に進んでください。')

保存完了
  models/model_v2.pkl             学習済みモデル
  models/species_centroids.npy    樹種平均スペクトル (13, 1555)
  models/species_centroid_ids.npy 樹種番号 [np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(8), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19)]

  アルゴリズム : Ridge回帰（類似度特徴量あり）
  alpha        : 1100
  CV RMSE      : 17.8367 ± 6.2885

次は predicting_v2.ipynb に進んでください。
